# Question-Adaptive Visual Token Selection for Vision-Language Models

**Experimental notebook**: CLIP ViT-B/16 (frozen) + question encoder (frozen) + question-conditioned *selector* + projector + Qwen2.5-0.5B-Instruct (frozen + LoRA), evaluated on VQAv2.

| Part | Sections | Content |
|---|---|---|
| I. Framing | 1 – 3 | Motivation, theoretical background, environment |
| II. Data | 4 – 5 | VQAv2, official metric, image preprocessing |
| III. Encoders | 6 – 8 | CLIP vision, question encoder, token analysis |
| IV. Selection | 9 – 13 | Scorers, cross-attention, fixed Top-K, conditioned Top-K, dynamic budget |
| V. VLM | 14 – 16 | Projector, integration into Qwen, full forward pass |
| VI. Training | 17 – 18 | LoRA, phases A → D |
| VII. Experiments | 19 – 21 | Baselines, benchmark, visualizations and ablations |
| VIII. Conclusion | 22 – 23 | Results, limitations, future work |

**Notebook rules**

1. Always run **section 3** first: it defines `cfg`, `DEVICE`, `DTYPE` and the utilities.
2. `DEBUG = True` (section 3.6) runs the whole pipeline on ~200 examples in a few minutes. Validate each section in DEBUG before switching to full mode.
3. Each section writes files to persistent storage; after a disconnection, we **reload** instead of recomputing (`load_or_compute`, section 3.7).
4. No number is written by hand in this notebook: all results are measured.

---
# Part I: framing

## Section 1. Introduction and motivation

### 1.1 The problem

A LLaVA-style VLM turns the image into a sequence of visual tokens, then feeds it to the LLM together with the question. With CLIP ViT-B/16 at 224×224, that is **196 visual tokens for every image**, whether the question is about the whole scene ("How many people are there?") or about a single object ("What color is the car?").

Yet the cost of the LLM grows with the sequence length $T$: linearly for the projections and the MLP, quadratically for attention. In our setup, visual tokens make up about **three quarters** of the sequence. Most of them are useless for answering a localized question.

### 1.2 The hypothesis

> A **question-conditioned** selection of visual tokens can sharply reduce the number of tokens sent to the LLM without degrading accuracy much. It does better than a selection that ignores the question, for the same number of tokens.

### 1.3 Research questions

| | Question | How we answer it |
|---|---|---|
| **RQ1** | At equal K, does a question-conditioned selection beat one that ignores the question? | Compare Question-Conditioned Top-K and Fixed Top-K (4 variants) for each K (sections 19 and 20) |
| **RQ2** | How low can K go before a significant degradation? | Accuracy = f(K) curve, gap to Full Tokens with confidence intervals |
| **RQ3** | Does the dynamic budget dominate the fixed budget on the accuracy / tokens curve? | Position of the "dynamic budget" point relative to the Pareto front of fixed Top-K |
| **RQ4** | Do FLOP savings translate into real latency savings? | Benchmark measured from pixels: ViT + selector + prefill + decoding |

### 1.4 Architecture

🔒 = frozen, 🔥 = trained

```
                     ┌──────────────────────────┐
  Image 3×224×224 ──►│ 🔒 CLIP ViT-B/16          │── V : 196×768 (penultimate layer, without CLS) ───┐
                     └──────────────────────────┘                                                   │
                                                                                                     ▼
                     ┌──────────────────────────┐                                  ┌──────────────────────────┐
  Question ─────────►│ 🔒 Question encoder       │── Q : M×512,  q̄ : 512 ─────────►│ 🔥 Selector               │── s : 196
                     │   (CLIP text)             │                                  │   (cross-attention, MLP…) │     │
                     └──────────────────────────┘                                  └──────────────────────────┘     │
                                                                                                                     ▼
                                                                                    ┌──────────────────────────┐
                                                                                    │ 🔥 Budget (fixed K or     │── K ∈ {24,49,98,147,196}
                                                                                    │    budget head)           │
                                                                                    └──────────────────────────┘
                                                                                                     │
                     Top-K(s, K) → sort indices (spatial order) → gather(V)  ◄──────────────────────┘
                                         │
                                         ▼  V_K : K×768
                     ┌──────────────────────────┐
                     │ 🔥 Projector (MLP)        │── K×896
                     └──────────────────────────┘
                                         │
                                         ▼
     [prompt] <|vision_start|> K visual tokens <|vision_end|> [question]   (T ≈ K + 40 tokens)
                                         │
                                         ▼
                     ┌──────────────────────────┐
                     │ 🔒 Qwen2.5-0.5B + 🔥 LoRA │──►  "red"
                     └──────────────────────────┘
```

### 1.5 The running example

We follow a single example throughout the notebook: a COCO image with a red car, a house and some people, and the question **"What color is the car?"**. In each section, we will print the actual shapes for this example.

| Step | What the example becomes | Shape | Section |
|---|---|---|---|
| Raw image | COCO photo | H×W×3 | 4 |
| Preprocessing | Resized and normalized image | 3×224×224 | 5 |
| CLIP ViT | 1 CLS + 196 patches (14×14 grid) | 197×768 | 6 |
| CLS removal | Visual tokens | V : 196×768 | 6 |
| Question | "What color is the car?" tokenized | Q : M×512, q̄ : 512 | 7 |
| Selector | One score per patch: those of the car are high | s : 196 | 9 – 12 |
| Budget | Simple question, localized object → K = 24 | K | 13 |
| Top-K | The 24 patches of the car, in spatial order | 24×768 | 11 – 12 |
| Projector | Translation into Qwen's space | 24×896 | 14 |
| LLM sequence | Prompt + 24 visual tokens + question | T ≈ 64 × 896 | 15 |
| Qwen + LoRA | Generation | "red" | 16 – 18 |
| Evaluation | Comparison with the 10 human answers | acc ∈ [0, 1] | 4, 20 |

## Section 2. Theoretical background

The bare minimum to get started. The rest (cross-attention in detail, STE, Gumbel, LoRA, RoPE, causal attention) is introduced in the section where we need it.

### 2.1 Attention

We start from three matrices obtained by linear projection of the input: the queries $Q \in \mathbb{R}^{n \times d_k}$, the keys $K \in \mathbb{R}^{m \times d_k}$ and the values $V \in \mathbb{R}^{m \times d_v}$.

$$
\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V
$$

* $QK^\top \in \mathbb{R}^{n \times m}$ measures how well each query "matches" each key.
* The **softmax** is applied row by row: each query distributes a mass of 1 over the $m$ keys.
* The output of each query is a **weighted average of the values**.

**Why divide by $\sqrt{d_k}$?** If the components of $q$ and $k$ are independent, with mean 0 and variance 1, then $q \cdot k = \sum_{i=1}^{d_k} q_i k_i$ has variance $d_k$. Without normalization, the logits become very large as $d_k$ grows, the softmax saturates (almost one-hot) and its gradients vanish. Dividing by $\sqrt{d_k}$ brings the variance back to 1.

**Multi-head.** We split $d_{model}$ into $h$ heads of dimension $d_k = d_{model}/h$. Each head computes its own attention, then we concatenate and project:

$$
\mathrm{MHA}(X) = \mathrm{Concat}(\mathrm{head}_1, \dots, \mathrm{head}_h)\, W^O,
\qquad \mathrm{head}_i = \mathrm{Attention}(X W_i^Q, X W_i^K, X W_i^V)
$$

* **Self-attention**: $Q$, $K$ and $V$ come from the same sequence (the ViT, the LLM).
* **Cross-attention**: $Q$ comes from one sequence, $K$ and $V$ from another. This is what our *selector* will use (section 10): the words of the question "look at" the patches.

### 2.2 The Vision Transformer (ViT-B/16)

1. **Patch splitting.** The $224 \times 224 \times 3$ image is split into a $14 \times 14$ grid of $16 \times 16$-pixel patches, i.e. $N = 196$ patches. Each patch contains $16 \cdot 16 \cdot 3 = 768$ values.
2. **Patch embedding.** Each flattened patch is linearly projected: $x_p E$, with $E \in \mathbb{R}^{768 \times D}$ and $D = 768$. In practice, this is a convolution with kernel 16 and stride 16.
3. **CLS token.** A learned vector $x_{cls}$ is prepended; it serves as a global summary of the image.
4. **Positional embedding.** Attention is permutation-invariant: without position information, the model would not know where each patch is. We add a learned position vector $E_{pos} \in \mathbb{R}^{197 \times D}$.

$$
z_0 = [\,x_{cls};\ x_1 E;\ x_2 E;\ \dots;\ x_{196} E\,] + E_{pos} \in \mathbb{R}^{197 \times 768}
$$

5. **12 Transformer blocks** (pre-LayerNorm): $z'_\ell = z_{\ell-1} + \mathrm{MHA}(\mathrm{LN}(z_{\ell-1}))$, then $z_\ell = z'_\ell + \mathrm{MLP}(\mathrm{LN}(z'_\ell))$.

For us, each patch yields a **visual token**. We will take the outputs of the **penultimate layer**, without the CLS: $V \in \mathbb{R}^{196 \times 768}$. The justification is given in section 6.

### 2.3 The LLaVA-style VLM architecture

$$
\underbrace{\text{image}}_{3\times224\times224}
\xrightarrow{\ \text{ViT}\ } Z_v \in \mathbb{R}^{196 \times 768}
\xrightarrow{\ \text{projector}\ } H_v \in \mathbb{R}^{196 \times 896}
\xrightarrow{\ \text{concat with the text}\ } [\,H_{prompt};\ H_v;\ H_{question}\,]
\xrightarrow{\ \text{LLM}\ } \text{answer}
$$

* The **projector** translates the ViT space into the LLM embedding space. The two spaces were learned separately, and nothing aligns them at the start.
* The visual tokens are given to the LLM through `inputs_embeds`, **in place of** word embeddings.
* The loss is the cross-entropy **only on the answer tokens**: $\mathcal{L} = -\sum_{t \in \text{answer}} \log p_\theta(y_t \mid y_{<t}, H_v, \text{question})$.

### 2.4 Visual token pruning

**The cost.** For one LLM block, the prefill of a sequence of length $T$ costs roughly

$$
\mathrm{FLOPs}_{\text{block}} \approx \underbrace{2\,T\,(2d^2 + 2 d\, d_{kv})}_{\text{projections } Q,K,V,O} \;+\; \underbrace{4\,T^2 d}_{\text{scores and aggregation}} \;+\; \underbrace{6\,T\, d\, d_{ff}}_{\text{SwiGLU MLP}}
$$

For Qwen2.5-0.5B ($d = 896$, $d_{ff} = 4864$), the term linear in $T$ dominates at the lengths considered. Going from $T \approx 236$ (196 visual tokens) to $T \approx 64$ ($K = 24$) therefore divides the prefill cost by about $3.7$. This theoretical calculation will be **verified by a measurement** in section 20 (RQ4).

**The principle.** We compute a score $s_i$ for each visual token, then keep the $K$ best:

$$
\mathcal{S} = \mathrm{TopK}(s, K), \qquad V_K = \{\, v_i : i \in \mathcal{S} \,\} \in \mathbb{R}^{K \times 768}
$$

Two choices structure the whole project:

| | **Question-independent** | **Question-conditioned** |
|---|---|---|
| **Fixed K** | Random, grid, CLS attention, learned scorer without the question (section 11) | Selector A / B / C + Top-K (section 12) |
| **Dynamic K** | (out of scope) | Mass threshold, entropy, budget head (section 13) |

Pruning can also be done **inside the LLM** (FastV drops tokens after the first layers based on the LLM's attention). We prune **before** the LLM, so that the savings apply to all of its layers. FastV remains an optional comparison (section 19).

## Section 3. Environment and configuration

**Goal**: a reproducible notebook that can restart after a disconnection without recomputing everything.

| Subsection | Role |
|---|---|
| 3.1 | Detect the environment (Colab, Kaggle, local) and choose persistent storage |
| 3.2 | Install the libraries and **pin their versions** in a lock file |
| 3.3 | Import and print the versions |
| 3.4 | Check the GPU and choose **fp16 or bf16** |
| 3.5 | Fix all the seeds |
| 3.6 | Central configuration + `DEBUG` flag |
| 3.7 | Resume (`load_or_compute`) and shape-printing utilities |
| 3.8 | Section checks |

### 3.1 Runtime environment and persistent storage

Two kinds of storage, with two different roles:

* **Persistent storage** (`PROJECT_DIR`): Google Drive on Colab. This is where we put whatever is expensive to recompute: split ID lists, feature cache, checkpoints, results.
* **Fast local disk** (`FAST_DIR`): this is where we put the **raw COCO images**. Reading thousands of small files from Drive is very slow; we would rather re-download them at each session.

Determinism-related environment variables must be set **before the first torch import**: that is why they are in the very first code cell.

In [1]:
# 3.1 — Runtime environment and storage
import os
import sys
import platform
from pathlib import Path

# Must be set BEFORE the first `import torch` (cuBLAS determinism, see 3.5)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

PROJECT_NAME = "Question_Adaptive_VLM"


def detect_env() -> str:
    """Returns 'colab', 'kaggle' or 'local'."""
    try:
        import google.colab  # noqa: F401
        return "colab"
    except ImportError:
        pass
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
        return "kaggle"
    return "local"


ENV = detect_env()

if os.environ.get("VLM_PROJECT_DIR"):            # manual override possible
    PROJECT_DIR = Path(os.environ["VLM_PROJECT_DIR"])
    FAST_DIR = PROJECT_DIR / "fast"
elif ENV == "colab":
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        PROJECT_DIR = Path("/content/drive/MyDrive") / PROJECT_NAME
    except Exception as e:                          # e.g. Colab kernel driven from VS Code
        print(f"⚠️ Drive not mounted ({type(e).__name__}: {e})")
        print("   → local storage: it will be LOST at the end of the session.")
        PROJECT_DIR = Path("/content") / PROJECT_NAME
    FAST_DIR = Path("/content/fast")
elif ENV == "kaggle":
    PROJECT_DIR = Path("/kaggle/working") / PROJECT_NAME   # kept only via "Save Version"
    FAST_DIR = Path("/tmp/fast")
else:
    PROJECT_DIR = Path.cwd() / "work"
    FAST_DIR = PROJECT_DIR / "fast"

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
FAST_DIR.mkdir(parents=True, exist_ok=True)

print(f"Environment     : {ENV}")
print(f"Python          : {platform.python_version()}  ({platform.system()})")
print(f"Persistent storage (PROJECT_DIR) : {PROJECT_DIR}")
print(f"Fast disk          (FAST_DIR)    : {FAST_DIR}")

Environment     : local
Python          : 3.13.2  (Windows)
Persistent storage (PROJECT_DIR) : c:\Users\riahi\Desktop\Question-Adaptive Visual Token Selection for Efficient VLM\work
Fast disk          (FAST_DIR)    : c:\Users\riahi\Desktop\Question-Adaptive Visual Token Selection for Efficient VLM\work\fast


### 3.2 Installation and version pinning

* **First run**: we install the missing libraries, then write their exact versions to `requirements.lock.txt` (on persistent storage).
* **Subsequent runs**: we reinstall **exactly** those versions. The notebook therefore does not change behavior if Colab updates its packages.

⚠️ We **never** reinstall `torch`: on Colab and Kaggle, it is built for the machine's CUDA version. Its version is only recorded in the lock. Locally, install it yourself from [pytorch.org](https://pytorch.org) with the right CUDA version.

If this cell **modified** an already-installed library, restart the kernel and run again from 3.1.

In [2]:
# 3.2 — Installation and version pinning
import subprocess
import importlib.util
from importlib.metadata import version, PackageNotFoundError

LOCK_FILE = PROJECT_DIR / "requirements.lock.txt"

# pip name → import name
LIBS = {
    "transformers": "transformers",
    "peft": "peft",
    "accelerate": "accelerate",
    "datasets": "datasets",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "pillow": "PIL",
    "tqdm": "tqdm",
    "scikit-learn": "sklearn",
}


def pip_install(*args: str) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--root-user-action=ignore", *args])


if importlib.util.find_spec("torch") is None:
    raise RuntimeError("PyTorch is missing: install the appropriate CUDA build (https://pytorch.org), then rerun.")

if LOCK_FILE.exists():
    print(f"📌 Pinned versions found → installing from {LOCK_FILE.name}")
    pip_install("-r", str(LOCK_FILE))
else:
    missing = [pkg for pkg, mod in LIBS.items() if importlib.util.find_spec(mod) is None]
    if missing:
        print("Installing:", ", ".join(missing))
        pip_install(*missing)
    lines = [f"# torch=={version('torch')}  (not reinstalled, depends on CUDA)"]
    lines += [f"{pkg}=={version(pkg)}" for pkg in LIBS]
    LOCK_FILE.write_text("\n".join(lines) + "\n")
    print(f"📌 Versions pinned in {LOCK_FILE}")

print(LOCK_FILE.read_text())

Installing: peft, accelerate
📌 Versions pinned in c:\Users\riahi\Desktop\Question-Adaptive Visual Token Selection for Efficient VLM\work\requirements.lock.txt
# torch==2.10.0  (not reinstalled, depends on CUDA)
transformers==5.0.0
peft==0.21.3
accelerate==1.15.0
datasets==4.5.0
numpy==2.2.2
pandas==2.2.3
matplotlib==3.10.0
pillow==11.1.0
tqdm==4.67.1
scikit-learn==1.8.0



### 3.3 Imports and versions

All the "heavy" imports of the notebook are centralized here. The following sections will only import what is specific to them.

In [3]:
# 3.3 — Imports and versions
import json
import math
import time
import random
import hashlib
from dataclasses import dataclass, asdict, fields

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

import transformers
import peft
import accelerate
import datasets

transformers.logging.set_verbosity_error()   # hides verbose loading warnings

for lib in (torch, transformers, peft, accelerate, datasets, np, pd):
    print(f"{lib.__name__:<13}{lib.__version__}")
print(f"{'CUDA (torch)':<13}{torch.version.cuda}")

torch        2.10.0+cpu
transformers 5.0.0
peft         0.21.3
accelerate   1.15.0
datasets     4.5.0
numpy        2.2.2
pandas       2.2.3
CUDA (torch) None


### 3.4 GPU and compute precision

| Format | Exponent bits | Mantissa bits | Range | Note |
|---|---|---|---|---|
| fp32 | 8 | 23 | ±3.4·10³⁸ | Reference, slow and memory-hungry |
| **fp16** | 5 | 10 | ±65,504 | Overflow risk: a `GradScaler` is needed for training |
| **bf16** | 8 | 7 | ±3.4·10³⁸ | Same range as fp32, less precise, no `GradScaler` needed |

bf16 is only **hardware-accelerated** from the Ampere architecture onward (compute capability ≥ 8.0: A100, L4, RTX 30xx/40xx). On a **T4** (capability 7.5), we use fp16.

⚠️ Pitfall: in recent PyTorch versions, `torch.cuda.is_bf16_supported()` may return `True` on a T4, because bf16 is **emulated** there (hence slow). We test the compute capability directly instead.

In [4]:
# 3.4 — GPU and precision choice
def setup_device():
    if not torch.cuda.is_available():
        print("⚠️ No GPU detected: only DEBUG mode is realistic on CPU.")
        return torch.device("cpu"), torch.float32, {}

    idx = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(idx)
    major, minor = torch.cuda.get_device_capability(idx)
    free_b, total_b = torch.cuda.mem_get_info(idx)

    native_bf16 = major >= 8
    dtype = torch.bfloat16 if native_bf16 else torch.float16
    if native_bf16:
        torch.backends.cuda.matmul.allow_tf32 = True   # accelerated fp32 matmuls (Ampere+)
        torch.backends.cudnn.allow_tf32 = True

    info = {
        "name": props.name,
        "capability": f"{major}.{minor}",
        "total_gb": round(total_b / 1024**3, 1),
        "free_gb": round(free_b / 1024**3, 1),
        "n_sm": props.multi_processor_count,
        "native_bf16": native_bf16,
    }
    return torch.device("cuda", idx), dtype, info


DEVICE, DTYPE, GPU_INFO = setup_device()
USE_GRAD_SCALER = DTYPE == torch.float16    # essential in fp16, useless in bf16

for k, v in GPU_INFO.items():
    print(f"{k:<12}{v}")
print(f"{'device':<12}{DEVICE}")
print(f"{'dtype':<12}{DTYPE}")
print(f"{'GradScaler':<12}{USE_GRAD_SCALER}")

⚠️ No GPU detected: only DEBUG mode is realistic on CPU.
device      cpu
dtype       torch.float32
GradScaler  False


### 3.5 Seeds and determinism

There are **four independent generators** to seed: `random` (Python), NumPy, torch CPU and torch CUDA. On top of that come two less visible sources of non-determinism:

* **non-deterministic CUDA algorithms** (some cuDNN/cuBLAS convolutions and reductions). We disable them with `cudnn.deterministic` and `torch.use_deterministic_algorithms`. `warn_only=True` reports operations that have no deterministic version instead of crashing;
* **DataLoader workers**, each of which has its own seed. We control them with `worker_init_fn=seed_worker` and `generator=make_generator(seed)`.

Deterministic mode slightly slows down some operations. We will **disable** it for the latency benchmark (section 20).

In [5]:
# 3.5 — Seeds
def set_seed(seed: int, deterministic: bool = True) -> None:
    """Seeds every source of randomness: Python, NumPy, torch (CPU and CUDA)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)              # CPU
    torch.cuda.manual_seed_all(seed)     # all GPUs (no effect without a GPU)
    torch.backends.cudnn.deterministic = deterministic
    torch.backends.cudnn.benchmark = not deterministic
    torch.use_deterministic_algorithms(deterministic, warn_only=True)


def seed_worker(worker_id: int) -> None:
    """Pass as `worker_init_fn` to the DataLoader: each worker derives its seed from torch's."""
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_generator(seed: int) -> torch.Generator:
    """Pass as `generator=` to the DataLoader: makes the shuffle order reproducible."""
    g = torch.Generator()
    g.manual_seed(seed)
    return g


SEED = 42
set_seed(SEED)
print(f"Global seed set: {SEED}")

Global seed set: 42


### 3.6 Central configuration

All the project's hyperparameters live in **a single object** `cfg`. No section should hard-code a number: it reads `cfg`.

A few choices worth understanding:

* **The K buckets**: $\{24, 49, 98, 147, 196\} \approx 196 \times \{\tfrac18, \tfrac14, \tfrac12, \tfrac34, 1\}$. They cover the range from 12.5% to 100% of the tokens.
* **`vision_feature_layer = -2`**: the penultimate layer of the ViT (justification in section 6).
* **LoRA r = 16 on the 7 projections** of each Qwen block: this gives about 8.8M trainable parameters (we will check this in section 17).
* **Subset sizes** are expressed in **number of images**, because the train / dev / test split is done by `image_id` (an image has ~5 questions on average in VQAv2).
* **`run_name`**: DEBUG artifacts and full-mode artifacts go into **separate folders**. Otherwise, `load_or_compute` would reload, in full mode, a cache computed on 40 images.

The learning rates and loss weights are **starting values**; they will be tuned on the dev set in section 18.

In [6]:
# 3.6 — Central configuration
DEBUG = True    # ← True: ~200 examples, a few minutes. False: full run.


@dataclass
class Config:
    # ---------------- General
    debug: bool = DEBUG
    seed: int = SEED
    eval_seeds: tuple = (0, 1, 2)            # 3 seeds for the key comparisons
    env: str = ENV
    device: str = str(DEVICE)
    dtype: str = str(DTYPE).replace("torch.", "")

    # ---------------- Paths (filled in __post_init__)
    project_dir: str = str(PROJECT_DIR)
    fast_dir: str = str(FAST_DIR)
    run_name: str = ""
    data_dir: str = ""          # VQAv2 JSON, splits, DataFrame (shared between DEBUG and full)
    image_dir: str = ""         # raw COCO images (fast disk)
    run_dir: str = ""           # everything that depends on the run
    cache_dir: str = ""         # CLIP features, CLS attention, teacher attention
    ckpt_dir: str = ""          # projector, LoRA, selector, budget head
    results_dir: str = ""       # raw result CSVs
    fig_dir: str = ""           # figures

    # ---------------- Models (Hugging Face)
    clip_name: str = "openai/clip-vit-base-patch16"
    distilbert_name: str = "distilbert-base-uncased"
    llm_name: str = "Qwen/Qwen2.5-0.5B-Instruct"

    # ---------------- Dimensions
    image_size: int = 224
    patch_size: int = 16
    grid_size: int = 14
    n_patches: int = 196
    d_vision: int = 768
    d_text: int = 512               # CLIP text
    d_distilbert: int = 768
    d_llm: int = 896                # Qwen2.5-0.5B
    vision_feature_layer: int = -2  # penultimate layer of the ViT
    max_question_len: int = 32      # in CLIP tokens (max context: 77)
    max_answer_len: int = 8         # in Qwen tokens

    # ---------------- Token budgets
    k_buckets: tuple = (24, 49, 98, 147, 196)
    k_fixed: tuple = (24, 49, 98, 147)
    k_blind: int = 0                # blind baseline: no visual token

    # ---------------- Selector and budget (starting values)
    selector_dim: int = 256
    selector_heads: int = 8
    gumbel_scale_start: float = 1.0
    gumbel_scale_end: float = 0.0
    lambda_distill: float = 0.5
    budget_rho: float = 0.9         # cumulative mass threshold (training-free budget)

    # ---------------- Projector
    projector_type: str = "mlp"     # "linear" or "mlp"
    projector_hidden: int = 896

    # ---------------- LoRA
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    lora_targets: tuple = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")

    # ---------------- Training
    lr_projector: float = 1e-3
    lr_lora: float = 2e-4
    lr_selector: float = 3e-4
    lr_budget: float = 1e-3
    batch_size: int = 16
    grad_accum: int = 4
    weight_decay: float = 0.0
    warmup_ratio: float = 0.03
    max_grad_norm: float = 1.0
    epochs_a: int = 1               # phase A: projector only
    epochs_b: int = 1               # phase B: LoRA
    epochs_c: int = 1               # phase C: selector
    epochs_d: int = 3               # phase D: budget head
    eval_every: int = 500           # in optimization steps
    num_workers: int = 2

    # ---------------- Data (in number of images)
    n_train_images: int = 10_000    # ≈ 54k questions
    n_dev_images: int = 1_000
    n_test_images: int = 2_000      # drawn from val2014 (official test set is unannotated)

    # ---------------- Evaluation and benchmark
    n_bootstrap: int = 1_000
    bench_warmup: int = 10
    bench_reps: int = 50
    bench_batch_sizes: tuple = (1, 8, 32)

    def __post_init__(self):
        if self.debug:
            self.n_train_images, self.n_dev_images, self.n_test_images = 40, 10, 10   # ≈ 200 + 50 + 50 questions
            self.batch_size, self.grad_accum = 4, 1
            self.epochs_a = self.epochs_b = self.epochs_c = self.epochs_d = 1
            self.eval_every = 20
            self.eval_seeds = (0,)
            self.n_bootstrap = 100
            self.bench_warmup, self.bench_reps = 2, 5
            self.num_workers = 0
        self.run_name = "debug" if self.debug else "full"
        root = Path(self.project_dir)
        self.data_dir = str(root / "data")
        self.image_dir = str(Path(self.fast_dir) / "coco")
        self.run_dir = str(root / "runs" / self.run_name)
        for name in ("cache", "ckpt", "results", "fig"):
            setattr(self, f"{name}_dir", str(Path(self.run_dir) / name))

    # --- utilities
    def path(self, kind: str, *parts: str) -> Path:
        """cfg.path('cache', 'clip_train.pt') → Path to the file."""
        return Path(getattr(self, f"{kind}_dir")).joinpath(*parts)

    def make_dirs(self) -> None:
        for f in fields(self):
            if f.name.endswith("_dir"):
                Path(getattr(self, f.name)).mkdir(parents=True, exist_ok=True)

    def validate(self) -> None:
        assert self.image_size // self.patch_size == self.grid_size, "image_size / patch_size ≠ grid_size"
        assert self.grid_size ** 2 == self.n_patches, "grid_size² ≠ n_patches"
        assert list(self.k_buckets) == sorted(set(self.k_buckets)), "k_buckets must be sorted, without duplicates"
        assert all(1 <= k <= self.n_patches for k in self.k_buckets), "K outside [1, n_patches]"
        assert self.k_buckets[-1] == self.n_patches, "the last bucket must be Full Tokens"
        assert set(self.k_fixed) <= set(self.k_buckets), "k_fixed must be included in k_buckets"
        assert self.d_llm % 64 == 0 and self.selector_dim % self.selector_heads == 0
        assert 0 < self.budget_rho < 1 and 0 <= self.lambda_distill

    def to_dict(self) -> dict:
        return {k: (list(v) if isinstance(v, tuple) else v) for k, v in asdict(self).items()}


cfg = Config()
cfg.validate()
cfg.make_dirs()

# Save the run config, with a warning if it changed since the last execution
cfg_file = cfg.path("run", "config.json")
if cfg_file.exists():
    old = json.loads(cfg_file.read_text())
    changed = {k: (old.get(k), v) for k, v in cfg.to_dict().items() if old.get(k) != v}
    if changed:
        print("⚠️ Config changed since the last execution of this run:")
        for k, (a, b) in changed.items():
            print(f"   {k}: {a} → {b}")
        print("   Already-cached artifacts may have been produced with the old config.")
cfg_file.write_text(json.dumps(cfg.to_dict(), indent=2, ensure_ascii=False))

print(f"Mode: {'DEBUG' if cfg.debug else 'FULL'}   (run_name = '{cfg.run_name}')")
print(f"Train / dev / test images: {cfg.n_train_images} / {cfg.n_dev_images} / {cfg.n_test_images}")
print(f"K buckets: {cfg.k_buckets}   ({[f'{k / cfg.n_patches:.0%}' for k in cfg.k_buckets]})")
print(f"Effective batch: {cfg.batch_size} × {cfg.grad_accum} = {cfg.batch_size * cfg.grad_accum}")
print(f"Run dir: {cfg.run_dir}")

Mode: DEBUG   (run_name = 'debug')
Train / dev / test images: 40 / 10 / 10
K buckets: (24, 49, 98, 147, 196)   (['12%', '25%', '50%', '75%', '100%'])
Effective batch: 4 × 1 = 4
Run dir: c:\Users\riahi\Desktop\Question-Adaptive Visual Token Selection for Efficient VLM\work\runs\debug


### 3.7 Resume and display utilities

**`load_or_compute`** is the mechanism that makes the notebook restartable: if the file exists, we reload it; otherwise, we compute it and then save it.

Writing is **atomic**: we first write to `file.tmp`, then rename. If Colab disconnects in the middle of a save, an incomplete `.tmp` is left behind, but never a fake "finished" file that would be wrongly reloaded.

**`show`** prints the shape, dtype and a few statistics of a tensor. We will use it at each step of the running example.

In [7]:
# 3.7 — Resume and display utilities
def load_or_compute(path, compute_fn, save_fn=None, load_fn=None, force: bool = False):
    """Reloads `path` if it exists; otherwise runs `compute_fn()`, saves the result and returns it.

    Default: torch.save / torch.load. Pass save_fn(obj, path) and load_fn(path) for another format.
    """
    path = Path(path)
    save_fn = save_fn or (lambda obj, p: torch.save(obj, p))
    load_fn = load_fn or (lambda p: torch.load(p, map_location="cpu", weights_only=False))

    if path.exists() and not force:
        print(f"↩️  Reloaded: {path.name}")
        return load_fn(path)

    t0 = time.perf_counter()
    obj = compute_fn()
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    save_fn(obj, tmp)
    tmp.replace(path)                       # atomic rename
    size_mb = path.stat().st_size / 1024**2
    print(f"💾 Computed and saved: {path.name}  ({time.perf_counter() - t0:.1f} s, {size_mb:.1f} MB)")
    return obj


def show(name: str, x) -> None:
    """Prints shape, dtype, device and statistics of a tensor (or the length of a sequence)."""
    if isinstance(x, torch.Tensor):
        msg = f"{name:<28} {str(tuple(x.shape)):<18} {str(x.dtype).replace('torch.', ''):<9} {x.device}"
        if x.is_floating_point() and x.numel() > 0:
            xf = x.detach().float()
            msg += f"  min={xf.min():+.3f} max={xf.max():+.3f} mean={xf.mean():+.3f} std={xf.std():.3f}"
        print(msg)
    elif hasattr(x, "__len__"):
        print(f"{name:<28} len={len(x)}  type={type(x).__name__}")
    else:
        print(f"{name:<28} {x!r}")


def count_params(module: nn.Module) -> tuple[int, int]:
    """(total parameters, trainable parameters)."""
    total = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    return total, trainable


def gpu_mem(tag: str = "") -> None:
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1024**3
        peak = torch.cuda.max_memory_allocated() / 1024**3
        print(f"[GPU {tag}] allocated = {alloc:.2f} GB | peak = {peak:.2f} GB")


show("example", torch.randn(196, 768))

example                      (196, 768)         float32   cpu  min=-4.590 max=+4.629 mean=+0.000 std=1.005


### 3.8 Section 3 checks

1. The GPU is detected (otherwise, a warning: only DEBUG mode is realistic).
2. **Reproducibility**: two draws with the same seed are identical, and a different seed gives a different draw. We test the 4 generators, plus the order of a shuffled `DataLoader`.
3. The run folders are writable.
4. `load_or_compute` computes the first time and reloads the second time.

In [8]:
# 3.8 — Checks
from torch.utils.data import DataLoader


def draw(seed: int) -> dict:
    set_seed(seed)
    out = {
        "python": random.random(),
        "numpy": np.random.rand(3).round(8).tolist(),
        "torch_cpu": torch.rand(3).tolist(),
        "randperm": torch.randperm(cfg.n_patches)[:8].tolist(),
        "dataloader": [b.tolist() for b in DataLoader(range(20), batch_size=5, shuffle=True,
                                                     generator=make_generator(seed),
                                                     worker_init_fn=seed_worker)],
    }
    if torch.cuda.is_available():
        out["torch_cuda"] = torch.rand(3, device="cuda").tolist()
    return out


checks = {}

# 1. GPU
checks["GPU detected"] = torch.cuda.is_available()

# 2. Reproducibility
a, b, c = draw(cfg.seed), draw(cfg.seed), draw(cfg.seed + 1)
checks["same seed → same draw"] = a == b
checks["different seed → different draw"] = all(a[k] != c[k] for k in a)
print("Draw (seed 42):", {k: a[k] for k in ("python", "randperm")})

# 3. Writing
probe = cfg.path("run", ".write_test")
probe.write_text("ok")
checks["run_dir is writable"] = probe.read_text() == "ok"
probe.unlink()

# 4. load_or_compute
test_file = cfg.path("cache", "_test_load_or_compute.pt")
test_file.unlink(missing_ok=True)
x1 = load_or_compute(test_file, lambda: torch.arange(10))
x2 = load_or_compute(test_file, lambda: torch.zeros(10))       # must NOT be executed
checks["load_or_compute reloads the cache"] = torch.equal(x1, x2)
test_file.unlink()

set_seed(cfg.seed)   # start again from a clean state

print()
for name, ok in checks.items():
    print(f"{'✅' if ok else '❌'} {name}")

critical = {k: v for k, v in checks.items() if k != "GPU detected"}
assert all(critical.values()), "A critical check failed: do not move on to section 4."
if not checks["GPU detected"]:
    print("\n⚠️ No GPU: OK for DEBUG, but sections 6 onward will be very slow.")
print("\nSection 3 validated → we can move on to section 4.")

Draw (seed 42): {'python': 0.6394267984578837, 'randperm': [110, 187, 41, 55, 24, 105, 19, 182]}
💾 Computed and saved: _test_load_or_compute.pt  (0.0 s, 0.0 MB)
↩️  Reloaded: _test_load_or_compute.pt

❌ GPU detected
✅ same seed → same draw
✅ different seed → different draw
✅ run_dir is writable
✅ load_or_compute reloads the cache

⚠️ No GPU: OK for DEBUG, but sections 6 onward will be very slow.

Section 3 validated → we can move on to section 4.


---
# Part II: the data

## Section 4. Preparing the VQAv2 dataset

**Goal**: a clean table of (image, question, 10 answers) triplets, train / dev / test splits that share no image, and a reliable implementation of the official metric.

| Subsection | Role |
|---|---|
| 4.1 | Download the VQAv2 question and annotation JSON files (train2014 and val2014) |
| 4.2 | Join questions and annotations on `question_id` → one DataFrame |
| 4.3 | Explore: answer types, frequent question types, question length, answer long tail |
| 4.4 | Split train / dev by `image_id` (from train2014), test from val2014; save the ID lists |
| 4.5 | Download only the COCO 2014 images of the subset |
| 4.6 | Official answer normalization (punctuation, numbers, articles, contractions) |
| 4.7 | Official VQA Accuracy (average over the 10 subsets of 9 annotators) |
| 4.8 | Training target `multiple_choice_answer`; save the normalized DataFrame |
| 4.9 | Section checks |

The COCO images are downloaded **after** the split (4.5): we only fetch the images that are actually in our subset, not the whole of COCO 2014.

### 4.1 Downloading the VQAv2 annotations

**Goal**: get the four official VQAv2 JSON files onto persistent storage (`cfg.data_dir`, shared between DEBUG and full mode).

**What is in these files:**

* **questions** file: one entry per question, with `question_id`, `image_id` and `question`;
* **annotations** file, with the same `question_id`: the **10 human answers** (`answers`), the most frequent one (`multiple_choice_answer`), the `answer_type` (yes/no, number, other) and the `question_type` (the first words of the question, e.g. "what color is the").

The images are **not** in these files: they come from COCO 2014, and we download them in 4.5.

**Why val2014 is our test set**: the answers for the official test-dev / test-std splits are hidden behind an evaluation server. val2014 is annotated, so it serves as our test set. Our dev set is carved out of train2014 (4.4).

Same principle as `load_or_compute`: if a JSON is already there, nothing is downloaded. Each file is written to `.tmp` and then renamed, so an interrupted download never leaves behind a file that looks complete. The URL and file names are fixed properties of the dataset, not hyperparameters, so they stay in this cell rather than in `cfg`.

In [9]:
# 4.1 — Download the VQAv2 JSON files
import shutil
import urllib.request
import zipfile

VQA_DIR = Path(cfg.data_dir) / "vqav2"
VQA_DIR.mkdir(parents=True, exist_ok=True)

VQA_BASE_URL = "https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa"

# key → (official archive, JSON file inside that archive)
VQA_FILES = {
    "train_questions":   ("v2_Questions_Train_mscoco.zip",   "v2_OpenEnded_mscoco_train2014_questions.json"),
    "train_annotations": ("v2_Annotations_Train_mscoco.zip", "v2_mscoco_train2014_annotations.json"),
    "val_questions":     ("v2_Questions_Val_mscoco.zip",     "v2_OpenEnded_mscoco_val2014_questions.json"),
    "val_annotations":   ("v2_Annotations_Val_mscoco.zip",   "v2_mscoco_val2014_annotations.json"),
}


def fetch_vqa_json(archive: str, member: str) -> Path:
    """Downloads `archive`, extracts `member`, then deletes the archive. Does nothing if the JSON is already there."""
    json_path = VQA_DIR / member
    if json_path.exists():
        print(f"↩️  Already there: {member}")
        return json_path

    t0 = time.perf_counter()

    # 1. Download the archive (to .tmp, then rename)
    zip_path = VQA_DIR / archive
    tmp_zip = zip_path.with_name(zip_path.name + ".tmp")
    urllib.request.urlretrieve(f"{VQA_BASE_URL}/{archive}", tmp_zip)
    tmp_zip.replace(zip_path)

    # 2. Extract only the JSON we need (to .tmp, then rename)
    tmp_json = json_path.with_name(json_path.name + ".tmp")
    with zipfile.ZipFile(zip_path) as zf, zf.open(member) as src, open(tmp_json, "wb") as dst:
        shutil.copyfileobj(src, dst)
    tmp_json.replace(json_path)          # the JSON only appears once it is complete

    # 3. The archive is no longer needed
    zip_path.unlink()

    print(f"⬇️  Downloaded and extracted: {member}  ({time.perf_counter() - t0:.1f} s)")
    return json_path


VQA_JSON = {key: fetch_vqa_json(archive, member) for key, (archive, member) in VQA_FILES.items()}

print()
for key, p in VQA_JSON.items():
    print(f"{key:<18} {p.stat().st_size / 1024**2:>7.1f} MB   {p.name}")

⬇️  Downloaded and extracted: v2_OpenEnded_mscoco_train2014_questions.json  (35.0 s)
⬇️  Downloaded and extracted: v2_mscoco_train2014_annotations.json  (53.2 s)
⬇️  Downloaded and extracted: v2_OpenEnded_mscoco_val2014_questions.json  (9.4 s)
⬇️  Downloaded and extracted: v2_mscoco_val2014_annotations.json  (61.9 s)

train_questions       40.0 MB   v2_OpenEnded_mscoco_train2014_questions.json
train_annotations    339.0 MB   v2_mscoco_train2014_annotations.json
val_questions         19.3 MB   v2_OpenEnded_mscoco_val2014_questions.json
val_annotations      163.8 MB   v2_mscoco_val2014_annotations.json


### 4.2 Joining questions and annotations

**Goal**: one row per question, with everything we need later: `coco_split`, `image_id`, `question`, `question_id`, `question_type`, `answer_type`, `multiple_choice_answer` and `answers` (the list of the 10 human answers).

**Concept: the join key.** Questions and annotations live in two separate files, linked by `question_id`. We use `merge(..., validate="one_to_one")`: pandas raises an error if a `question_id` appears twice on either side, which would otherwise **silently duplicate rows**. We also check that both files give the same `image_id` for each question.

**What we drop**: `answer_confidence` ("yes" / "maybe" / "no") and `answer_id`. The official VQA metric does not use them.

**Caching**: parsing the annotations JSON is the slow part. The joined DataFrame is saved with `load_or_compute` in `cfg.data_dir`, not in the run's `cache_dir`, because it does not depend on DEBUG. We save it as a pickle rather than Parquet, because the `answers` column holds lists and `pyarrow` is not in our lock file.

No answer normalization yet: that comes in 4.6. For the running example, we only count the questions that are exactly "What color is the car?". The actual image is chosen once the images are downloaded (4.5).

In [ ]:
# 4.2 — Join questions and annotations into one DataFrame
import gc


def load_split(split: str) -> pd.DataFrame:
    """Reads the questions and annotations of one split ('train' or 'val') and joins them on question_id."""
    with open(VQA_JSON[f"{split}_questions"], encoding="utf-8") as f:
        questions = pd.DataFrame(json.load(f)["questions"])        # columns: image_id, question, question_id

    with open(VQA_JSON[f"{split}_annotations"], encoding="utf-8") as f:
        raw = json.load(f)["annotations"]
    annotations = pd.DataFrame({
        "question_id":            [a["question_id"] for a in raw],
        "image_id_ann":           [a["image_id"] for a in raw],     # only used to cross-check, then dropped
        "question_type":          [a["question_type"] for a in raw],
        "answer_type":            [a["answer_type"] for a in raw],
        "multiple_choice_answer": [a["multiple_choice_answer"] for a in raw],
        # Only the answer strings are kept: the official metric ignores `answer_confidence`
        "answers":                [[x["answer"] for x in a["answers"]] for a in raw],
    })
    del raw

    # validate="one_to_one" raises an error if a question_id appears twice on either side
    df = questions.merge(annotations, on="question_id", how="inner", validate="one_to_one")
    assert len(df) == len(questions) == len(annotations), "some questions have no annotation (or the reverse)"
    assert (df["image_id"] == df["image_id_ann"]).all(), "image_id differs between questions and annotations"
    df = df.drop(columns="image_id_ann")
    df.insert(0, "coco_split", f"{split}2014")                     # needed to build the image path (4.5)
    return df


def build_vqa_dataframe() -> pd.DataFrame:
    df = pd.concat([load_split("train"), load_split("val")], ignore_index=True)
    gc.collect()                                                    # free the raw JSON (several GB of Python objects)
    return df


# Independent of DEBUG / full → stored in data_dir, not in the run's cache_dir
vqa_df = load_or_compute(
    cfg.path("data", "vqav2_joined.pkl"),
    build_vqa_dataframe,
    save_fn=lambda obj, p: obj.to_pickle(p),
    load_fn=pd.read_pickle,
)

assert vqa_df["question_id"].is_unique, "question_id is not unique across train2014 and val2014"

print()
show("vqa_df", vqa_df)
print(vqa_df.dtypes.to_string())

print("\nPer COCO split:")
print(vqa_df.groupby("coco_split").agg(questions=("question_id", "size"),
                                       images=("image_id", "nunique")).to_string())
print("\nNumber of human answers per question:", vqa_df["answers"].map(len).value_counts().to_dict())

display(vqa_df.head(3))

# Running example: how many questions are exactly "What color is the car?", and what do people answer?
example_rows = vqa_df[vqa_df["question"].str.lower() == "what color is the car?"]
print(f'Running example: {len(example_rows)} questions are exactly "What color is the car?"')
print("Most frequent multiple_choice_answer:")
print(example_rows["multiple_choice_answer"].value_counts().head(5).to_string())

## Section 5. Image preprocessing

🚧 *To be built cell by cell: see `PLAN.md`, section 5.*

---
# Part III: the encoders

## Section 6. The visual encoder (CLIP ViT-B/16)

🚧 *To be built cell by cell: see `PLAN.md`, section 6.*

## Section 7. The question encoder

🚧 *To be built cell by cell: see `PLAN.md`, section 7.*

## Section 8. Visual token analysis

🚧 *To be built cell by cell: see `PLAN.md`, section 8.*

---
# Part IV: token selection

## Section 9. Scoring mechanisms

🚧 *To be built cell by cell: see `PLAN.md`, section 9.*

## Section 10. Cross-attention implementation

🚧 *To be built cell by cell: see `PLAN.md`, section 10.*

## Section 11. Fixed Top-K (without the question)

🚧 *To be built cell by cell: see `PLAN.md`, section 11.*

## Section 12. Question-Conditioned Top-K

🚧 *To be built cell by cell: see `PLAN.md`, section 12.*

## Section 13. The dynamic budget

🚧 *To be built cell by cell: see `PLAN.md`, section 13.*

---
# Part V: assembling the VLM

## Section 14. The projector

🚧 *To be built cell by cell: see `PLAN.md`, section 14.*

## Section 15. Integration into the LLM

🚧 *To be built cell by cell: see `PLAN.md`, section 15.*

## Section 16. The full VLM forward pass

🚧 *To be built cell by cell: see `PLAN.md`, section 16.*

---
# Part VI: training

## Section 17. LoRA and PEFT

🚧 *To be built cell by cell: see `PLAN.md`, section 17.*

## Section 18. Training and validation

🚧 *To be built cell by cell: see `PLAN.md`, section 18.*

---
# Part VII: experiments

## Section 19. Baseline experiments

🚧 *To be built cell by cell: see `PLAN.md`, section 19.*

## Section 20. Evaluation and benchmark

🚧 *To be built cell by cell: see `PLAN.md`, section 20.*

## Section 21. Visualizations and ablations

🚧 *To be built cell by cell: see `PLAN.md`, section 21.*

---
# Part VIII: conclusion

## Section 22. Results and discussion

🚧 *To be built cell by cell: see `PLAN.md`, section 22.*

## Section 23. Limitations and future work

🚧 *To be built cell by cell: see `PLAN.md`, section 23.*